In [ ]:
import os
import io
import pickle
import warnings
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import lightgbm as lgb
import yfinance as yf
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────
# CONFIGURACIÓN CENTRAL  ← Toca solo aquí
# ─────────────────────────────────────────────────────────────────

CONFIG = {
    # Dónde guardar modelos y predicciones
    # Opciones: "local" | "gcs"
    "storage_mode": "local",

    # Si storage_mode == "local" (Colab / desarrollo)
    "local_model_dir":      "/content/drive/MyDrive/modelos_lgbm",
    "local_output_dir":     "/content/drive/MyDrive/predicciones",

    # Si storage_mode == "gcs" (Cloud Run / producción)
    # Instala: pip install google-cloud-storage
    "gcs_bucket":           "tu-bucket-nombre",
    "gcs_model_prefix":     "modelos_lgbm/",
    "gcs_output_prefix":    "predicciones/",

    # Quantiles para el rango ético (p5-p95 más seguro que p10-p90)
    "quantile_low":  0.05,
    "quantile_high": 0.95,

    # Mínimo de días históricos para entrenar
    "min_history_days": 60,

    # Días de histórico del Brent a descargar
    "brent_start_date": "2022-01-01",
}

# ─────────────────────────────────────────────────────────────────
# 1. STORAGE: LOCAL ↔ GOOGLE CLOUD STORAGE
# ─────────────────────────────────────────────────────────────────

def _get_gcs_client():
    from google.cloud import storage
    return storage.Client()

def save_bytes(data: bytes, path: str):
    """Guarda bytes en local o GCS según CONFIG."""
    if CONFIG["storage_mode"] == "gcs":
        client = _get_gcs_client()
        bucket = client.bucket(CONFIG["gcs_bucket"])
        blob = bucket.blob(CONFIG["gcs_output_prefix"] + path)
        blob.upload_from_string(data)
        print(f"  ✓ Guardado en GCS: gs://{CONFIG['gcs_bucket']}/{blob.name}")
    else:
        full_path = os.path.join(CONFIG["local_output_dir"], path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, "wb") as f:
            f.write(data)
        print(f"  ✓ Guardado en local: {full_path}")

def load_bytes(path: str):
    """Carga bytes desde local o GCS. Retorna None si no existe."""
    if CONFIG["storage_mode"] == "gcs":
        from google.cloud import storage
        client = _get_gcs_client()
        bucket = client.bucket(CONFIG["gcs_bucket"])
        blob = bucket.blob(CONFIG["gcs_model_prefix"] + path)
        if not blob.exists():
            return None
        return blob.download_as_bytes()
    else:
        full_path = os.path.join(CONFIG["local_model_dir"], path)
        if not os.path.exists(full_path):
            return None
        with open(full_path, "rb") as f:
            return f.read()

def save_model(models, feature_cols, station, fuel):
    filename = _model_filename(station, fuel)
    data = pickle.dumps({"models": models, "feature_cols": feature_cols})
    # Para modelos siempre usamos el directorio de modelos
    if CONFIG["storage_mode"] == "gcs":
        from google.cloud import storage
        client = _get_gcs_client()
        bucket = client.bucket(CONFIG["gcs_bucket"])
        blob = bucket.blob(CONFIG["gcs_model_prefix"] + filename)
        blob.upload_from_string(data)
        print(f"  ✓ Modelo guardado en GCS: {filename}")
    else:
        os.makedirs(CONFIG["local_model_dir"], exist_ok=True)
        path = os.path.join(CONFIG["local_model_dir"], filename)
        with open(path, "wb") as f:
            f.write(data)
        print(f"  ✓ Modelo guardado en local: {path}")

def load_model(station, fuel):
    """Carga modelo .pkl si existe. Retorna (models, feature_cols) o (None, None)."""
    filename = _model_filename(station, fuel)
    raw = load_bytes(filename) if CONFIG["storage_mode"] == "gcs" else None

    if CONFIG["storage_mode"] == "local":
        path = os.path.join(CONFIG["local_model_dir"], filename)
        if os.path.exists(path):
            with open(path, "rb") as f:
                raw = f.read()

    if raw is None:
        print(f"  → No existe modelo guardado para {station} | {fuel}")
        return None, None

    data = pickle.loads(raw)
    print(f"  ✓ Modelo cargado desde {'GCS' if CONFIG['storage_mode'] == 'gcs' else 'local'}")
    return data["models"], data["feature_cols"]

def _model_filename(station, fuel):
    fuel_clean = fuel.replace(" ", "_").replace("/", "-")
    return f"{station}_{fuel_clean}.pkl"

def save_predictions(pred_df: pd.DataFrame, run_date: str):
    """
    Guarda las predicciones en CSV y Parquet.
    Nombre: predicciones_YYYY-MM-DD.csv / .parquet
    """
    for fmt in ["csv", "parquet"]:
        filename = f"predicciones_{run_date}.{fmt}"
        if fmt == "csv":
            buf = io.BytesIO()
            pred_df.to_csv(buf, index=False, decimal=",", sep=";")
            data = buf.getvalue()
        else:
            buf = io.BytesIO()
            pred_df.to_parquet(buf, index=False)
            data = buf.getvalue()
        save_bytes(data, filename)

# ─────────────────────────────────────────────────────────────────
# 2. CARGA DE DATOS HISTÓRICOS
# ─────────────────────────────────────────────────────────────────

def load_raw_data(path_pattern: str) -> pd.DataFrame:
    files = glob.glob(path_pattern)
    print(f"Total archivos: {len(files)}")
    dfs = []
    for f in files:
        try:
            d = pd.read_parquet(f)
            if "IDEESS" in d.columns:
                dfs.append(d)
        except Exception as e:
            print(f"Error leyendo {f}: {e}")
    df = pd.concat(dfs, ignore_index=True)

    def parse_price(col):
        return (
            df[col].astype(str)
            .replace("", np.nan)
            .str.replace(",", ".", regex=False)
            .astype(float)
        )

    for c in ["Precio Gasolina 95 E5", "Precio Gasoleo A"]:
        if c in df.columns:
            df[c] = parse_price(c)

    df["fecha"] = pd.to_datetime(df["fecha_registro"], unit="ms").dt.normalize()
    df["IDEESS"] = df["IDEESS"].astype(str).str.strip()
    return df

# ─────────────────────────────────────────────────────────────────
# 3. PRECIO DEL BRENT (yfinance, activado por defecto)
# ─────────────────────────────────────────────────────────────────

def load_brent(start_date: str = None) -> pd.DataFrame:
    """
    Descarga el precio del Brent desde Yahoo Finance.
    Compatible con yfinance >= 0.2.x (MultiIndex columns).
    """
    start = start_date or CONFIG["brent_start_date"]
    print(f"  Descargando Brent desde {start}...")
    try:
        import yfinance as yf
        raw = yf.download("BZ=F", start=start, progress=False, auto_adjust=True)

        if raw.empty:
            print("  ⚠️  No se pudo descargar el Brent. Continuando sin él.")
            return None

        # yfinance >= 0.2 devuelve MultiIndex: ("Close", "BZ=F")
        # Aplanamos para obtener solo "Close"
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)

        brent_df = (
            raw[["Close"]]
            .reset_index()
            .rename(columns={"Date": "fecha", "Close": "brent"})
        )
        brent_df["fecha"] = pd.to_datetime(brent_df["fecha"]).dt.normalize()

        # Rellenar todos los días (fines de semana sin cotización)
        all_days = pd.date_range(brent_df["fecha"].min(), brent_df["fecha"].max(), freq="D")
        brent_df = (
            brent_df.set_index("fecha")
            .reindex(all_days)
            .rename_axis("fecha")
            .reset_index()
        )
        brent_df["brent"] = brent_df["brent"].ffill()

        print(f"  ✓ Brent cargado: {len(brent_df)} días | "
              f"último: {brent_df['fecha'].max().date()} = {brent_df['brent'].iloc[-1]:.2f} USD/barril")
        return brent_df

    except Exception as e:
        print(f"  ⚠️  Error descargando Brent: {e}. Continuando sin él.")
        return None

# ─────────────────────────────────────────────────────────────────
# 4. SERIE TEMPORAL
# ─────────────────────────────────────────────────────────────────

def build_ts(df, station, fuel):
    d = df[df["IDEESS"] == station].copy()
    ts = d.groupby("fecha")[fuel].mean().sort_index()
    all_days = pd.date_range(start=ts.index.min(), end=ts.index.max(), freq="D")
    ts = ts.reindex(all_days).ffill().bfill()
    ts = ts.reset_index().rename(columns={"index": "fecha", fuel: "y"})
    return ts

# ─────────────────────────────────────────────────────────────────
# 5. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────

def add_features(ts, brent_df=None):
    df_feat = ts.copy().sort_values("fecha").reset_index(drop=True)

    # Temporales
    df_feat["dia_semana"]       = df_feat["fecha"].dt.dayofweek
    df_feat["es_fin_de_semana"] = (df_feat["dia_semana"] >= 5).astype(int)
    df_feat["semana_anio"]      = df_feat["fecha"].dt.isocalendar().week.astype(int)
    df_feat["mes"]              = df_feat["fecha"].dt.month
    df_feat["trimestre"]        = df_feat["fecha"].dt.quarter
    df_feat["dia_mes"]          = df_feat["fecha"].dt.day

    # Lags
    for lag in [1, 7, 14, 21, 30]:
        df_feat[f"lag_{lag}"] = df_feat["y"].shift(lag)

    # Rolling
    for w in [7, 14, 30]:
        df_feat[f"roll_mean_{w}"] = df_feat["y"].shift(1).rolling(w).mean()
    df_feat["roll_std_7"] = df_feat["y"].shift(1).rolling(7).std()

    # Diferencias
    df_feat["diff_1d"] = df_feat["y"].diff(1)
    df_feat["diff_7d"] = df_feat["y"].diff(7)

    # Brent
    if brent_df is not None:
        df_feat = df_feat.merge(brent_df[["fecha", "brent"]], on="fecha", how="left")
        df_feat["brent"] = df_feat["brent"].ffill()
        df_feat["lag_brent_1"]  = df_feat["brent"].shift(1)
        df_feat["lag_brent_7"]  = df_feat["brent"].shift(7)
        df_feat["diff_brent_7"] = df_feat["brent"].diff(7)   # tendencia Brent última semana

    return df_feat

BASE_FEATURES = [
    "dia_semana", "es_fin_de_semana", "semana_anio", "mes", "trimestre", "dia_mes",
    "lag_1", "lag_7", "lag_14", "lag_21", "lag_30",
    "roll_mean_7", "roll_mean_14", "roll_mean_30",
    "roll_std_7", "diff_1d", "diff_7d",
]
BRENT_FEATURES = ["brent", "lag_brent_1", "lag_brent_7", "diff_brent_7"]

def get_feature_cols(brent_df):
    return BASE_FEATURES + (BRENT_FEATURES if brent_df is not None else [])

# ─────────────────────────────────────────────────────────────────
# 6. ENTRENAMIENTO (p5 / p50 / p95)
# ─────────────────────────────────────────────────────────────────

LGBM_BASE = {
    "boosting_type":    "gbdt",
    "n_estimators":     400,
    "learning_rate":    0.04,
    "num_leaves":       31,
    "min_child_samples": 10,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "verbose":          -1,
    "n_jobs":           -1,
}

def train_lgbm(ts, brent_df=None):
    df_feat = add_features(ts, brent_df).dropna()
    feature_cols = [c for c in get_feature_cols(brent_df) if c in df_feat.columns]
    X, y = df_feat[feature_cols], df_feat["y"]

    models = {}
    for quantile, label in [
        (CONFIG["quantile_low"],  "low"),
        (0.50,                    "mid"),
        (CONFIG["quantile_high"], "high"),
    ]:
        m = lgb.LGBMRegressor(**{**LGBM_BASE, "objective": "quantile", "alpha": quantile})
        m.fit(X, y)
        models[label] = m

    return models, df_feat, feature_cols

# ─────────────────────────────────────────────────────────────────
# 7. FORECAST RECURSIVO 7 DÍAS
# ─────────────────────────────────────────────────────────────────

def forecast_7d(models, ts, df_feat, feature_cols, brent_df=None):
    last_date = ts["fecha"].max()
    all_y = list(df_feat["y"].values)
    all_brent = list(df_feat["brent"].values) if "brent" in df_feat.columns else []

    preds = {"fecha": [], "precio_predicho": [], "margen_min": [], "margen_max": []}

    for i in range(7):
        next_date = last_date + pd.Timedelta(days=i + 1)

        def lag(n):
            idx = len(all_y) - n
            return all_y[idx] if idx >= 0 else np.nan

        row = {
            "dia_semana":        next_date.dayofweek,
            "es_fin_de_semana":  int(next_date.dayofweek >= 5),
            "semana_anio":       int(next_date.isocalendar()[1]),
            "mes":               next_date.month,
            "trimestre":         next_date.quarter,
            "dia_mes":           next_date.day,
            "lag_1":   lag(1),  "lag_7":  lag(7),
            "lag_14":  lag(14), "lag_21": lag(21), "lag_30": lag(30),
            "roll_mean_7":  np.mean(all_y[-7:])  if len(all_y) >= 7  else np.nan,
            "roll_mean_14": np.mean(all_y[-14:]) if len(all_y) >= 14 else np.nan,
            "roll_mean_30": np.mean(all_y[-30:]) if len(all_y) >= 30 else np.nan,
            "roll_std_7":   np.std(all_y[-7:])   if len(all_y) >= 7  else np.nan,
            "diff_1d": all_y[-1] - all_y[-2] if len(all_y) >= 2 else 0,
            "diff_7d": all_y[-1] - all_y[-8] if len(all_y) >= 8 else 0,
        }

        # Brent features
        if brent_df is not None and len(all_brent) > 0:
            # Brent del día siguiente (último conocido si no hay cotización futura)
            future_brent = brent_df[brent_df["fecha"] <= next_date]
            row["brent"] = future_brent["brent"].iloc[-1] if len(future_brent) > 0 else all_brent[-1]
            row["lag_brent_1"] = all_brent[-1] if len(all_brent) >= 1 else np.nan
            row["lag_brent_7"] = all_brent[-7] if len(all_brent) >= 7 else np.nan
            row["diff_brent_7"] = (all_brent[-1] - all_brent[-7]) if len(all_brent) >= 7 else 0
            all_brent.append(row["brent"])

        X_pred = pd.DataFrame([row])[[c for c in feature_cols if c in row]]

        p_mid  = models["mid"].predict(X_pred)[0]
        p_low  = min(models["low"].predict(X_pred)[0], p_mid)
        p_high = max(models["high"].predict(X_pred)[0], p_mid)

        preds["fecha"].append(next_date)
        preds["precio_predicho"].append(round(p_mid, 3))
        preds["margen_min"].append(round(p_low, 3))
        preds["margen_max"].append(round(p_high, 3))

        all_y.append(p_mid)  # Predicción entra como lag del siguiente día

    return pd.DataFrame(preds)

# ─────────────────────────────────────────────────────────────────
# 8. PIPELINE PRINCIPAL
# ─────────────────────────────────────────────────────────────────

def predict_for_station_list(df, station_ids, fuels,
                              force_retrain=False, brent_df=None):
    all_preds = []
    run_date  = datetime.today().strftime("%Y-%m-%d")

    for st in station_ids:
        print(f"\n📍 Estación: {st}")
        for fuel in fuels:
            print(f"  ⛽ {fuel}")
            ts = build_ts(df, st, fuel)

            if len(ts) < CONFIG["min_history_days"]:
                print("  → Pocos datos, skip")
                continue

            # ── Cargar modelo guardado o entrenar desde cero ──
            models, feature_cols = None, None
            if not force_retrain:
                models, feature_cols = load_model(st, fuel)

            if models is None:
                print("  → Entrenando desde cero...")
                models, df_feat, feature_cols = train_lgbm(ts, brent_df)
            else:
                print("  → Reentrenando con datos nuevos...")
                models, df_feat, feature_cols = train_lgbm(ts, brent_df)

            save_model(models, feature_cols, st, fuel)

            df_feat = add_features(ts, brent_df).dropna()
            fc = forecast_7d(models, ts, df_feat, feature_cols, brent_df)
            fc["IDEESS"]     = st
            fc["fuel"]       = fuel
            fc["run_date"]   = run_date   # Fecha en que se generó la predicción
            all_preds.append(fc)
            print(f"  ✓ Predicciones generadas")

    pred_df = pd.concat(all_preds, ignore_index=True)

    # ── Guardar predicciones ──
    print(f"\n💾 Guardando predicciones...")
    save_predictions(pred_df, run_date)

    return pred_df

# ─────────────────────────────────────────────────────────────────
# 9. BACKTESTING (comparar con ARIMA)
# ─────────────────────────────────────────────────────────────────

def backtest(df, station, fuel, test_days=30, brent_df=None):
    ts = build_ts(df, station, fuel)
    n  = len(ts)
    train_ts = ts.iloc[: n - test_days].copy()
    test_ts  = ts.iloc[n - test_days :].copy()

    models, df_feat, feature_cols = train_lgbm(train_ts, brent_df)

    errors = []
    for i in range(0, test_days - 7, 7):
        chunk = pd.concat(
            [train_ts, test_ts.iloc[:i][["fecha", "y"]]], ignore_index=True
        ) if i > 0 else train_ts.copy()
        df_feat_chunk = add_features(chunk, brent_df).dropna()
        fc   = forecast_7d(models, chunk, df_feat_chunk, feature_cols, brent_df)
        real = test_ts.iloc[i : i + 7]["y"].values
        pred = fc["precio_predicho"].values[: len(real)]
        errors.extend(zip(real, pred))

    real_all = [e[0] for e in errors]
    pred_all = [e[1] for e in errors]

    mae  = mean_absolute_error(real_all, pred_all)
    rmse = np.sqrt(mean_squared_error(real_all, pred_all))
    cover = np.mean([
        fc["margen_min"].iloc[j] <= real_all[j] <= fc["margen_max"].iloc[j]
        for j in range(len(real_all))
        if j < len(fc)
    ]) * 100

    print(f"\n📊 Backtest {station} | {fuel}")
    print(f"  MAE:       {mae:.4f} €")
    print(f"  RMSE:      {rmse:.4f} €")
    print(f"  Cobertura rango: {cover:.1f}% de días dentro del rango")
    return mae, rmse

# ─────────────────────────────────────────────────────────────────
# 10. EJECUCIÓN
# ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── Configura aquí según dónde ejecutes ──────────────────────
    # Colab:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = "/content/drive/MyDrive/raw/*.parquet"

    # Cloud Run: los parquets vendrían de GCS
    # DATA_PATH = "/tmp/raw/*.parquet"  # descargados previamente de GCS
    # CONFIG["storage_mode"] = "gcs"
    # ─────────────────────────────────────────────────────────────

    df = load_raw_data(DATA_PATH)
    print(f"Shape total: {df.shape}")

    print("\n📈 Descargando precio del Brent...")
    brent_df = load_brent()

    station_ids = ["15158", "152", "10889"]
    fuels = ["Precio Gasolina 95 E5", "Precio Gasoleo A"]

    pred_df = predict_for_station_list(
        df,
        station_ids,
        fuels,
        force_retrain=True,  # ← True solo la primera vez
        brent_df=brent_df,
    )

    print("\n=== PREDICCIONES ===")
    print(pred_df.to_string(index=False))


Mounted at /content/drive
Total archivos: 741
Shape total: (8277332, 43)

📈 Descargando precio del Brent...
  Descargando Brent desde 2022-01-01...
  ✓ Brent cargado: 1517 días | último: 2026-02-27 = 72.48 USD/barril

📍 Estación: 15158
  ⛽ Precio Gasolina 95 E5
  → Entrenando desde cero...
  ✓ Modelo guardado en local: /content/drive/MyDrive/modelos_lgbm/15158_Precio_Gasolina_95_E5.pkl
  ✓ Predicciones generadas
  ⛽ Precio Gasoleo A
  → Entrenando desde cero...
  ✓ Modelo guardado en local: /content/drive/MyDrive/modelos_lgbm/15158_Precio_Gasoleo_A.pkl
  ✓ Predicciones generadas

📍 Estación: 152
  ⛽ Precio Gasolina 95 E5
  → Entrenando desde cero...
  ✓ Modelo guardado en local: /content/drive/MyDrive/modelos_lgbm/152_Precio_Gasolina_95_E5.pkl
  ✓ Predicciones generadas
  ⛽ Precio Gasoleo A
  → Entrenando desde cero...
  ✓ Modelo guardado en local: /content/drive/MyDrive/modelos_lgbm/152_Precio_Gasoleo_A.pkl
  ✓ Predicciones generadas

📍 Estación: 10889
  ⛽ Precio Gasolina 95 E5
  → En